In [7]:
import os
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def run_fsu_audit():
    # Define target desktop directory
    output_dir = r"C:\Users\Edumo\Desktop\Forensic Science Unit Analysis"
    os.makedirs(output_dir, exist_ok=True)

    dataset_file = os.path.join(output_dir, 'simulated_fsu_casework_audit.csv')
    
    if not os.path.exists(dataset_file):
        print(f"Error: Could not find '{dataset_file}'. Run 'generate_fsu_dataset.py' first.")
        return

    # Load evidence dataset
    df = pd.read_csv(dataset_file)

    # Flag SLA breaches: Court subpoena > 5 days OR Routine case > Target SLA
    df['SLA_Compliance_Breach'] = np.where(
        ((df['Priority_Level'] == 'Court Subpoena / Trial') & (df['Actual_Turnaround_Days'] > 5.0)) |
        ((df['Priority_Level'] != 'Court Subpoena / Trial') & (df['Actual_Turnaround_Days'] > df['Target_SLA_Days'])),
        1, 0
    )

    # Summarize performance metrics by discipline
    unit_summary = df.groupby('Assigned_Unit').agg(
        Total_Items=('Item_ID', 'count'),
        Avg_Target_SLA=('Target_SLA_Days', 'mean'),
        Avg_Actual_TAT=('Actual_Turnaround_Days', lambda x: round(x.mean(), 1)),
        Median_Actual_TAT=('Actual_Turnaround_Days', lambda x: round(x.median(), 1)),
        SLA_Breach_Rate=('SLA_Compliance_Breach', lambda x: f"{(x.mean()*100):.1f}%"),
        QC_Holds=('QC_Calibration_Passed', lambda x: (x == 0).sum()),
        CoC_Discrepancies=('Chain_of_Custody_Verified', lambda x: (x == 0).sum())
    ).reset_index()

    print("\n--- SPPD FORENSIC SERVICES UNIT PERFORMANCE SUMMARY ---")
    print(unit_summary.to_string(index=False))

    # Export summary table to target folder
    summary_file = os.path.join(output_dir, 'fsu_unit_performance_summary.csv')
    unit_summary.to_csv(summary_file, index=False)
    print(f"\nSaved unit breakdown to '{summary_file}'")

    # Build Excel report workbook
    wb = openpyxl.Workbook()

    navy_header = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    alt_row_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    
    font_header = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    font_title = Font(name="Calibri", size=16, bold=True, color="1F497D")
    font_subtitle = Font(name="Calibri", size=11, italic=True, color="595959")
    font_section = Font(name="Calibri", size=13, bold=True, color="1F497D")
    font_bold = Font(name="Calibri", size=11, bold=True)
    font_body = Font(name="Calibri", size=11)

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    # Sheet 1: Executive Summary
    ws_summary = wb.active
    ws_summary.title = "FSU Executive Summary"
    ws_summary.views.sheetView[0].showGridLines = True

    ws_summary["A1"] = "SAINT PAUL POLICE DEPARTMENT - FORENSIC SERVICES UNIT"
    ws_summary["A1"].font = font_title
    ws_summary["A2"] = "ISO/IEC 17020 Quality Assurance & Laboratory Casework Performance Audit"
    ws_summary["A2"].font = font_subtitle

    # Top KPI Metrics Cards
    kpis = [
        ("Total Cases Evaluated", f"{len(df):,}"),
        ("Chain of Custody Pass Rate", f"{(df['Chain_of_Custody_Verified'].mean()*100):.1f}%"),
        ("QC Calibration Pass Rate", f"{(df['QC_Calibration_Passed'].mean()*100):.1f}%"),
        ("Overall SLA Breach Rate", f"{(df['SLA_Compliance_Breach'].mean()*100):.1f}%")
    ]

    for idx, (title, val) in enumerate(kpis, start=1):
        col_pos = idx * 2 - 1
        ws_summary.cell(row=4, column=col_pos, value=title).font = Font(name="Calibri", size=9, bold=True, color="595959")
        ws_summary.cell(row=5, column=col_pos, value=val).font = Font(name="Calibri", size=14, bold=True, color="1F497D")

    ws_summary["A7"] = "Performance Metrics by Discipline"
    ws_summary["A7"].font = font_section

    table_headers = [
        "Forensic Unit", "Total Items", "Avg Target SLA (Days)", 
        "Avg Actual TAT (Days)", "SLA Breach Rate", "QC Calibration Holds"
    ]
    for col_idx, text in enumerate(table_headers, start=1):
        cell = ws_summary.cell(row=8, column=col_idx, value=text)
        cell.fill = navy_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal="center" if col_idx > 1 else "left", vertical="center")

    for row_idx, row in unit_summary.iterrows():
        current_row = 9 + row_idx
        ws_summary.cell(row=current_row, column=1, value=row['Assigned_Unit']).font = font_bold
        ws_summary.cell(row=current_row, column=2, value=row['Total_Items']).font = font_body
        ws_summary.cell(row=current_row, column=3, value=round(row['Avg_Target_SLA'], 1)).font = font_body
        ws_summary.cell(row=current_row, column=4, value=round(row['Avg_Actual_TAT'], 1)).font = font_body
        ws_summary.cell(row=current_row, column=5, value=row['SLA_Breach_Rate']).font = font_body
        ws_summary.cell(row=current_row, column=6, value=row['QC_Holds']).font = font_body
        
        for c in range(1, 7):
            cell = ws_summary.cell(row=current_row, column=c)
            cell.border = thin_border
            if row_idx % 2 == 1:
                cell.fill = alt_row_fill
            if c > 1:
                cell.alignment = Alignment(horizontal="center")

    # Sheet 2: Raw Casework Log
    ws_data = wb.create_sheet(title="Casework Master Data")
    ws_data.views.sheetView[0].showGridLines = True

    columns = list(df.columns)
    for col_idx, name in enumerate(columns, start=1):
        cell = ws_data.cell(row=1, column=col_idx, value=name)
        cell.fill = navy_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal="center", vertical="center")

    for row_idx, row in df.iterrows():
        for col_idx, col_name in enumerate(columns, start=1):
            val = row[col_name]
            if isinstance(val, pd.Timestamp):
                val = val.strftime('%Y-%m-%d %H:%M')
            cell = ws_data.cell(row=row_idx+2, column=col_idx, value=val)
            cell.font = font_body
            cell.border = thin_border
            if col_name in ['Chain_of_Custody_Verified', 'QC_Calibration_Passed', 'SLA_Compliance_Breach']:
                cell.alignment = Alignment(horizontal="center")

    # Auto-fit column widths
    for sheet in [ws_summary, ws_data]:
        for col in sheet.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            col_letter = get_column_letter(col[0].column)
            sheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

    excel_output = os.path.join(output_dir, 'fsu_performance_model.xlsx')
    wb.save(excel_output)
    print(f"Excel model saved to '{excel_output}'")

if __name__ == "__main__":
    run_fsu_audit()


--- SPPD FORENSIC SERVICES UNIT PERFORMANCE SUMMARY ---
 Assigned_Unit  Total_Items  Avg_Target_SLA  Avg_Actual_TAT  Median_Actual_TAT SLA_Breach_Rate  QC_Holds  CoC_Discrepancies
   Crime Scene          130             7.0             8.7                8.3           66.9%         4                  4
           DNA          254            25.0            26.2               26.4           65.0%         8                  3
Drug Chemistry          266            14.0            15.9               15.9           70.3%         8                  3
      Firearms          105            21.0            22.7               22.3           69.5%         2                  3
 Latent Prints          141            18.0            19.4               19.2           66.0%         7                  2
    Toxicology           57            20.0            20.9               20.3           61.4%         0                  1
Trace Evidence           47            30.0            31.6               3